In [1]:
!pip install optuna
# ── Imports & Setup ──────────────────────────────────────────
import pandas as pd
import numpy as np
import random
import warnings
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
# 減少 optuna 與 LightGBM 訓練時的洗版訊息
optuna.logging.set_verbosity(optuna.logging.WARNING)

def seed_everything(seed=2026):
    np.random.seed(seed)
    random.seed(seed)

seed_everything()

# ── 1. Data Loading ──────────────────────────────────────────
target_col = 'Irrigation_Need'
num_classes = 3

# 直接讀取當前目錄的檔案
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 預先保留 test id，以供最後產生提交檔使用，接著再將訓練用的 id 丟棄
test_ids = test['id']
train.drop(['id'], axis=1, inplace=True)
test.drop(['id'], axis=1, inplace=True)

# 目標變數(Target)轉換為數字
target2idx = {v: i for i, v in enumerate(train[target_col].unique())}
idx2target = {v: i for i, v in target2idx.items()}
train[target_col] = train[target_col].map(target2idx)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

# ── 2. Feature Engineering ───────────────────────────────────
M = train[NUMS].max()
def FE(df):
    for c in NUMS:
        # 提取數字特徵的特定位數
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')

        # 根據最大值做不同的四捨五入
        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)
    return df

train = FE(train)
test = FE(test)

# 刪除單一值的欄位 (沒有區分度的特徵)
DROP = [c for c in test.columns if test[c].nunique() == 1]
train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)

# 頻率編碼 (Frequency Encoding)
CATEGORY = CATS + [c for c in test.columns if 'digit' in c]
for c in CATEGORY:
    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + NUMS

# 計算樣本權重 (處理類別不平衡)
unique, counts = np.unique(train[target_col].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[target_col]])

# ── 3. Cross Validation & Training ───────────────────────────
def accuracy_score(t, p):
    if len(p.shape) == 2:
        p = np.argmax(p, axis=1)
    C = 3
    acc = 0.0
    for i in range(C):
        acc += np.sum((t == i) & (p == i)) / np.sum(t == i) / C
    return acc

def lgb_eval_metric(y_true, y_pred):
    score = accuracy_score(y_true, y_pred)
    return 'acc', score, True

lgb_params = {
    "n_estimators": 6000,
    'boosting_type': 'gbdt',
    'max_depth': 4,
    'num_leaves': 32,
    'learning_rate': 0.05,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'lambda_l1': 10,
    'lambda_l2': 10,
    'min_child_samples': 12,
    'random_state': 2026,
    'n_jobs': -1,
    'max_bin': 15000,
    'verbosity': -1,
    'subsample': 0.5,
    'subsample_for_bin': 100000,
    'subsample_freq': 1,
}

X = train.drop([target_col], axis=1)
y = train[target_col]
test_X = test.copy()

oof_preds = np.zeros((len(y), num_classes))
test_preds = np.zeros((len(test_X), num_classes))

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

print("開始訓練 LightGBM 模型...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"  --> Fold {fold+1}/{n_folds} 訓練中...")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    train_weights = sample_weights[train_idx]

    # Target Encoding
    te = TargetEncoder(target_type='multiclass', smooth='auto', cv=5, random_state=42)
    X_train_enc = pd.DataFrame(te.fit_transform(X_train[FEATURES], y_train), index=X_train.index)
    X_val_enc = pd.DataFrame(te.transform(X_val[FEATURES]), index=X_val.index)
    X_test_enc = pd.DataFrame(te.transform(test_X[FEATURES]), index=test_X.index)

    X_train = pd.concat([X_train, X_train_enc], axis=1).drop(CATS, axis=1)
    X_val = pd.concat([X_val, X_val_enc], axis=1).drop(CATS, axis=1)
    X_test = pd.concat([test_X, X_test_enc], axis=1).drop(CATS, axis=1)

    model = LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        eval_metric=lgb_eval_metric,
        callbacks=[early_stopping(250, verbose=False)]
    )

    oof_preds[val_idx] = model.predict_proba(X_val)
    test_preds += model.predict_proba(X_test) / n_folds

print(f"\n✅ OOF CV 準確率: {accuracy_score(y, oof_preds):.6f}")

# ── 4. Optuna Optimization ───────────────────────────────────
print("\n開始使用 Optuna 尋找最佳類別權重...")
def objective(trial):
    cw1 = trial.suggest_float('cw1', 0.5, 3.0)
    cw2 = trial.suggest_float('cw2', 0.5, 3.0)
    cw3 = trial.suggest_float('cw3', 0.5, 3.0)

    class_weights = np.array([cw1, cw2, cw3])
    adjusted_probs = oof_preds * class_weights
    adjusted_probs = adjusted_probs / adjusted_probs.sum(axis=1, keepdims=True)

    return accuracy_score(y, np.argmax(adjusted_probs, axis=1))

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=200, show_progress_bar=False)

print(f"✅ 最佳準確率: {study.best_value:.6f}")
print("最佳類別權重:")
print(f"  class_0 = {study.best_params['cw1']:.4f}")
print(f"  class_1 = {study.best_params['cw2']:.4f}")
print(f"  class_2 = {study.best_params['cw3']:.4f}")

# ── 5. Output Submission ─────────────────────────────────────
best_cw = np.array([study.best_params['cw1'], study.best_params['cw2'], study.best_params['cw3']])

# 套用最佳權重
final_test_probs = test_preds * best_cw
final_test_probs = final_test_probs / final_test_probs.sum(axis=1, keepdims=True)
final_test_preds = np.argmax(final_test_probs, axis=1)

# 直接利用前面的 test_ids 生成提交檔
sub = pd.DataFrame({
    'id': test_ids,
    target_col: final_test_preds
})
sub[target_col] = sub[target_col].map(idx2target)

sub.to_csv("lgb_with_TE.csv", index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 12.3 MB/s eta 0:00:00
開始訓練 LightGBM 模型...
  --> Fold 1/5 訓練中...
  --> Fold 2/5 訓練中...
  --> Fold 3/5 訓練中...
  --> Fold 4/5 訓練中...
  --> Fold 5/5 訓練中...

✅ OOF CV 準確率: 0.979227

開始使用 Optuna 尋找最佳類別權重...
✅ 最佳準確率: 0.979549
最佳類別權重:
  class_0 = 2.0414
  class_1 = 1.7017
  class_2 = 2.8194
